# Building Pong

Pong was one of the first video games: a ball bounces around the screen and you keep it in play with a paddle. In this notebook you'll build your own version in Python with the **pygame** library. Along the way you'll learn how a game loop works, and how to draw to the screen, read the keyboard, detect collisions and play sounds.

## What is pygame?

**pygame** is a real, widely-used Python library for making 2D games. The code here is the same code you'll find in other popular 2D Python games.

### The game surface

Calling `pygame.display.set_mode((width, height))` gives you a surface called the screen. Drawing functions like `screen.fill(...)` and `pygame.draw.rect(...)` change that surface in memory. When you call `pygame.display.flip()`, it pushes the finished frame to the display all at once. Drawing every shape and *then* flipping is what keeps the animation from flickering.

### The game loop

A game is a `while` loop where every pass through does the same four jobs, once per frame:

1. **Handle events** — responds to things that just happened, like a key press or mouse click, collected from `pygame.event.get()`.
2. **Update** — move the ball, move the paddle, check for collisions, change the score.
3. **Draw** — clear the screen and redraw everything in its new position.
4. **Flip and wait** — show the frame with `pygame.display.flip()`, then `clock.tick(60)` pauses just long enough to hold the loop at 60 frames per second, so the game runs at the same speed on every machine.

### Game sounds

`pygame.mixer.Sound(path)` loads a short sound effect once; calling `.play()` on it fires it off without pausing the game.

# Set Up The Game

This first cell does everything that only needs to happen *once*: start pygame, create the screen, and define the fixed values (the **constants**) the game uses — sizes, speeds, and colours. Pulling these out into named constants near the top makes them easy to find and tweak: change `PADDLE_SPEED` or `BALL_SPEED` here and the whole game feels different.

In [ ]:
import pygame
from codetto import samples

pygame.init()
pygame.mixer.init()

# Screen constants
WIDTH = 400 #@param {type:"slider", min:200, max: 700, step:10}
HEIGHT = 300 #@param {type:"slider", min:200, max: 700, step:10}

# Frame timing
FPS = 60 #@param
clock = pygame.time.Clock()

# Game Colors
BACKGROUND = (15, 15, 25)
PADDLE_COLOR = (240, 240, 240)
BALL_COLOR = (240, 90, 90)
TEXT_COLOR = (240, 240, 240)

# Paddle movement
PADDLE_WIDTH = 70 #@param {type:"slider", min:50, max: 100, step:5}
PADDLE_HEIGHT = 12 #@param {type:"slider", min:1, max: 20, step:1}
PADDLE_SPEED = 6 #@param {type:"slider", min:1, max: 20, step:1}
PADDLE_Y = HEIGHT - 30

# Ball
BALL_SIZE = 10 #@param {type:"slider", min:1, max: 20, step:1}
BALL_SPEED = 3 #@param {type:"slider", min:1, max: 20, step:1}

# Text Font
font = pygame.font.SysFont(None, 28)

# Sound Effects
bounce_sound = pygame.mixer.Sound(samples.Sounds.Pop)
miss_sound = pygame.mixer.Sound(samples.Sounds.Incorrect)

## Detecting collisions

The trickiest part of the game is knowing when the ball hits the paddle. pygame describes anything box-shaped with a **`pygame.Rect`**: the `x` of its left edge, the `y` of its top edge, then its width and height. `a.colliderect(b)` is `True` when two Rects overlap.

The game gives the paddle and the ball a Rect every frame. Even the round ball has one: the square that fits around it.

In pygame, `y` counts **down** from the top of the screen, so a bigger `y` is lower. Run the cell to see a ball falling towards a paddle that sits at `y=270`.

In [ ]:
paddle = pygame.Rect(150, 270, 70, 12)  # x, y, width, height

for ball_y in [240, 255, 262, 275]:
  ball = pygame.Rect(180, ball_y, 10, 10)
  print(f"Ball at y={ball_y}: touching the paddle? {ball.colliderect(paddle)}")

# The Game Loop

The setup cell created the pieces of the game, and `game_loop()` below puts them in motion. It starts by setting the values that *change* while you play: where the paddle and ball are, which way the ball is moving, and the score. Then it repeats the four jobs, once per frame. The numbered comments in the code match these steps:

1. **Handle events.** Codetto's game screen has no window to close, so there's nothing to handle, but pygame still expects the loop to collect its events.
2. **Update.**
   - *The paddle:* `pygame.key.get_pressed()` tells you which keys are held down right now. While ← or → is held, the paddle moves `PADDLE_SPEED` pixels each frame, and `max()` and `min()` stop it at the edges of the screen.
   - *The ball:* `ball_dx` and `ball_dy` are how far the ball moves across and down each frame. A bounce flips the sign of one of them: `ball_dx *= -1` sends the ball back the other way when it reaches a side wall, and `ball_dy *= -1` when it reaches the top.
   - *Hit or miss:* if the ball is falling (`ball_dy > 0`) and its Rect touches the paddle's, it bounces up and you score a point. If it falls past the bottom of the screen, you missed: the score goes back to 0 and the ball starts again from the middle.
3. **Draw.** Fill the screen with the background colour, then draw the paddle, the ball and the score in their new positions.
4. **Flip and wait.** `pygame.display.flip()` shows the finished frame, and `clock.tick(FPS)` waits so the game runs at the same speed on every computer.

This cell only *defines* the function. Nothing moves until the next cell calls it.

In [ ]:
def game_loop():
  paddle_x = WIDTH // 2 - PADDLE_WIDTH // 2
  ball_x, ball_y = WIDTH // 2, HEIGHT // 2
  ball_dx, ball_dy = BALL_SPEED, -BALL_SPEED
  score = 0
  
  running = True
  while running:
    # 1. Handle events (nothing to do here - there's no window to close)
    for event in pygame.event.get():
      pass
  
    # 2. Update: slide the paddle while an arrow key is held
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT]:
      paddle_x = max(paddle_x - PADDLE_SPEED, 0)
    if keys[pygame.K_RIGHT]:
      paddle_x = min(paddle_x + PADDLE_SPEED, WIDTH - PADDLE_WIDTH)
  
    # 2. Update: move the ball, and bounce it off the walls
    ball_x += ball_dx
    ball_y += ball_dy
    if ball_x <= 0 or ball_x >= WIDTH - BALL_SIZE:
      ball_dx *= -1
    if ball_y <= 0:
      ball_dy *= -1
  
    # 2. Update: bounce off the paddle (+1 point), or reset if the ball is missed
    paddle_rect = pygame.Rect(paddle_x, PADDLE_Y, PADDLE_WIDTH, PADDLE_HEIGHT)
    ball_rect = pygame.Rect(ball_x, ball_y, BALL_SIZE, BALL_SIZE)
    if ball_dy > 0 and ball_rect.colliderect(paddle_rect):
      ball_dy *= -1
      score += 1
      bounce_sound.play()
    elif ball_y > HEIGHT:
      miss_sound.play()
      score = 0
      ball_x, ball_y = WIDTH // 2, HEIGHT // 2
      ball_dx, ball_dy = BALL_SPEED, -BALL_SPEED
  
    # 3. Draw: clear the screen, then redraw everything in its new position
    screen.fill(BACKGROUND)
    pygame.draw.rect(screen, PADDLE_COLOR, paddle_rect)
    pygame.draw.ellipse(screen, BALL_COLOR, ball_rect)
    screen.blit(font.render(f"Score: {score}", True, TEXT_COLOR), (10, 10))
  
    # 4. Flip the finished frame to the display, then wait to hold a steady FPS
    pygame.display.flip()
    clock.tick(FPS)

# Let's Play!

The next cell creates the game screen and starts the loop. Move the paddle with the **←** and **→** arrow keys and keep the ball in play for as long as you can. Good luck!

To stop the game, click the **Stop** button on the cell.

In [ ]:
screen = pygame.display.set_mode((WIDTH, HEIGHT))
game_loop()

# Things to Try

- Change the sliders in the setup cell to make the game easier or harder, then run the setup cell and the game again.
- Add lives: give the player three misses before the game is over, and show the lives next to the score.
- Keep a high score that stays between games, and challenge your friends to beat it.
- Add "spin": when the ball hits the paddle, change `ball_dx` depending on which part of the paddle it hit.
- Add rows of blocks that disappear when the ball hits them, to recreate the arcade classic *Breakout*.
- Add background music: `play_async()` from `codetto.audio` starts a sound without pausing the game.

# Check Your Understanding